# Architecting Robust Production RAG Systems: Part 1

Companion notebook for the article **"Architecting Robust Production RAG Systems: Part 1"** 

## What you will build

1. **A query router** that inspects each query's shape and weights sparse vs dense retrieval accordingly - then unit-tests that decision like any other business logic.
2. **A four-stage pipeline** (pre-filter, retrieve, rerank, generate) that enforces seller and region boundaries structurally, with a defensive check at generation time.
3. **A latency budget** for a 1.2-second SLA, a profiler that finds the over-budget stage, and an assertion that turns the budget into a regression test.



In [16]:
!pip install transformers langchain-core langchain-community langchain-ollama langchain-openai langchain-huggingface pydantic faiss-cpu rank_bm25
!apt-get update -qq
!apt-get install -y -qq zstd

!curl -fsSL https://ollama.com/install.sh | sh

  Obtaining dependency information for transformers from https://files.pythonhosted.org/packages/41/c4/a12e1d9b387fb0c40a57116db82b457e8c771cb419163cda29204d74a595/transformers-5.15.1-py3-none-any.whl.metadata
  Obtaining dependency information for langchain-core from https://files.pythonhosted.org/packages/84/4c/508a90b9d2e3bd7738fd93cb2ac2178ce734662c75e69b3b81c445f1b360/langchain_core-1.6.0-py3-none-any.whl.metadata
  Obtaining dependency information for langchain-community from https://files.pythonhosted.org/packages/8f/39/5d97e42a3e95dc2a6d71b2f902a3fae71786131e11d01bddb604accb0ebe/langchain_community-0.4.2-py3-none-any.whl.metadata
  Obtaining dependency information for langchain-ollama from https://files.pythonhosted.org/packages/2c/b2/c2acb076590a98bee2816ed5f285e00df162a34238f9e276e175e14ebc35/langchain_ollama-1.1.0-py3-none-any.whl.metadata
  Obtaining dependency information for langchain-openai from https://files.pythonhosted.org/packages/35/61/6aa67be92298b40d51427cd4d6834a

In [17]:
import os
import getpass

# Set your OpenAI API Key here if you have one. This is optional.
openai_key = getpass.getpass("Enter your OpenAI API Key (leave empty to skip): ")
if openai_key:
    os.environ["OPENAI_API_KEY"] = openai_key

print("API key input prompts added.")

API key input prompts added.


## 1. Route queries before you retrieve

Our running example is the support search bot for **ShopStream**, a fictional marketplace.
Its traffic mixes several very different query shapes:

- `ORD-2024-55719 status` - an exact order-number lookup. Embedding similarity is actively harmful here: every order number is "similar" to every other order number in semantic space, but only one exact string is correct.
- `orders for jane.doe@example.com` - an exact email lookup, same story.
- `damaged package` - a terse keyword query where lexical and semantic signals both help.
- `why was my card charged twice for the same order` - a conversational question where meaning matters far more than exact tokens.

**Dense retrieval** (embedding similarity) handles the last shape well and the first two badly.
**Sparse retrieval** (BM25-style exact-token matching) is the mirror image.
A fixed 50/50 hybrid blend just averages their blind spots, so instead we route: a small function looks at each query and chooses the blend.

The router below is pure regex and heuristics on purpose - it costs microseconds and nothing else, and its interface (weights plus a `reason`) will not change if you later swap in a learned classifier.

In [18]:
import re

ORDER_ID_PATTERN = re.compile(r"\b(?:ORD|RMA)-\d{4}-\d{3,}\b", re.IGNORECASE)
EMAIL_PATTERN = re.compile(r"\b[\w.+-]+@[\w-]+\.[\w.]+\b")

def route_query(query: str) -> dict:
    """Return sparse/dense retrieval weights plus the routing reason.

    Weights are floats in [0, 1] and always sum to 1.0.
    """
    if ORDER_ID_PATTERN.search(query):
        return {"sparse_weight": 0.90, "dense_weight": 0.10, "reason": "order_id"}

    if EMAIL_PATTERN.search(query):
        return {"sparse_weight": 0.80, "dense_weight": 0.20, "reason": "email_lookup"}

    if len(query.split()) <= 3:
        return {"sparse_weight": 0.50, "dense_weight": 0.50, "reason": "short_query"}

    return {"sparse_weight": 0.20, "dense_weight": 0.80, "reason": "conversational"}


# --- routing logic is business logic: test it like business logic ---
routing_cases = [
    ("ORD-2024-55719 status",                              "order_id"),
    ("refund for rma-2023-0041 please",                    "order_id"),
    ("orders for jane.doe+promo@example.com",              "email_lookup"),
    ("damaged package",                                    "short_query"),
    ("why was my card charged twice for the same order",   "conversational"),
]

passed = 0
for query, expected in routing_cases:
    result = route_query(query)
    ok = result["reason"] == expected
    passed += ok
    print(f"[{'PASS' if ok else 'FAIL'}] {query!r} -> {result}")

print(f"\nRouting accuracy: {passed}/{len(routing_cases)}")

[PASS] 'ORD-2024-55719 status' -> {'sparse_weight': 0.9, 'dense_weight': 0.1, 'reason': 'order_id'}
[PASS] 'refund for rma-2023-0041 please' -> {'sparse_weight': 0.9, 'dense_weight': 0.1, 'reason': 'order_id'}
[PASS] 'orders for jane.doe+promo@example.com' -> {'sparse_weight': 0.8, 'dense_weight': 0.2, 'reason': 'email_lookup'}
[PASS] 'damaged package' -> {'sparse_weight': 0.5, 'dense_weight': 0.5, 'reason': 'short_query'}
[PASS] 'why was my card charged twice for the same order' -> {'sparse_weight': 0.2, 'dense_weight': 0.8, 'reason': 'conversational'}

Routing accuracy: 5/5


The harness prints a PASS/FAIL line per case and an accuracy score - all five pass with the heuristics above.
The `reason` field is what makes this testable: routing failures are silent (a broken pattern throws no exception, it just quietly degrades one slice of traffic), so the test set is the only thing standing between you and an invisible regression.

### Extensions

Some queries are not asking for better ranking - they are asking for a smaller haystack.
`"orders over $50"` or `"refunds between $20 and $100"` look conversational (long, no order ID, no email), but what they really need is a **metadata pre-filter** on the amount column *before* any search runs.

Extend the router as `route_query_v2` so that it:

1. Detects amount-bound language (`over $50`, `under $15`, `at least $99`) and amount-range language (`between $20 and $100`).
2. Returns `"requires_prefilter": True` plus a structured `"filter_hint"` - e.g. `{"type": "amount", "operator": "gt", "value": 50.0}` or `{"type": "amount_range", "min": 20.0, "max": 100.0}`. It does not need to be a full query parser; a structured hint is enough.
3. Falls through to the existing four-category logic, unchanged, for everything else (those results should carry `"requires_prefilter": False, "filter_hint": None`).
4. Passes the six-case harness below, which includes cases that must **not** trigger the pre-filter path - false positives are how routers rot.

In [19]:
import re

ORDER_ID_PATTERN = re.compile(r"\b(?:ORD|RMA)-\d{4}-\d{3,}\b", re.IGNORECASE)
EMAIL_PATTERN = re.compile(r"\b[\w.+-]+@[\w-]+\.[\w.]+\b")

AMOUNT_RANGE_PATTERN = re.compile(
    r"between\s+\$(\d+(?:\.\d{1,2})?)\s+and\s+\$(\d+(?:\.\d{1,2})?)", re.IGNORECASE
)
AMOUNT_BOUND_PATTERN = re.compile(
    r"\b(over|above|at least|under|below|at most)\s+\$(\d+(?:\.\d{1,2})?)", re.IGNORECASE
)
BOUND_OPERATORS = {
    "over": "gt", "above": "gt", "at least": "gte",
    "under": "lt", "below": "lt", "at most": "lte",
}

def route_query_v2(query: str) -> dict:
    """Extend the router with amount-bound and amount-range detection."""
    range_match = AMOUNT_RANGE_PATTERN.search(query)
    if range_match:
        low, high = (float(g) for g in range_match.groups())
        return {"sparse_weight": 0.25, "dense_weight": 0.75, "reason": "amount_range",
                "requires_prefilter": True,
                "filter_hint": {"type": "amount_range", "min": low, "max": high}}

    bound_match = AMOUNT_BOUND_PATTERN.search(query)
    if bound_match:
        phrase, value = bound_match.groups()
        # The operator comes from the matched phrase itself, not a second
        # scan of the query, so stray words elsewhere cannot flip it.
        return {"sparse_weight": 0.25, "dense_weight": 0.75, "reason": "amount_bound",
                "requires_prefilter": True,
                "filter_hint": {"type": "amount",
                                "operator": BOUND_OPERATORS[phrase.lower()],
                                "value": float(value)}}

    if ORDER_ID_PATTERN.search(query):
        return {"sparse_weight": 0.90, "dense_weight": 0.10, "reason": "order_id",
                "requires_prefilter": False, "filter_hint": None}

    if EMAIL_PATTERN.search(query):
        return {"sparse_weight": 0.80, "dense_weight": 0.20, "reason": "email_lookup",
                "requires_prefilter": False, "filter_hint": None}

    if len(query.split()) <= 3:
        return {"sparse_weight": 0.50, "dense_weight": 0.50, "reason": "short_query",
                "requires_prefilter": False, "filter_hint": None}

    return {"sparse_weight": 0.20, "dense_weight": 0.80, "reason": "conversational",
            "requires_prefilter": False, "filter_hint": None}

In [20]:
# Verification harness -- do not modify. Your router must pass all 6 cases.
prefilter_cases = [
    ("orders over $50",                                   True),
    ("refunds between $20 and $100",                      True),
    ("show purchases under $15 from my account",          True),
    ("ORD-2024-55719 status",                             False),
    ("why was my card charged twice for the same order",  False),
    ("track package",                                     False),
]

passed = 0
for query, expected in prefilter_cases:
    result = route_query_v2(query)
    ok = result["requires_prefilter"] == expected
    passed += ok
    print(f"[{'PASS' if ok else 'FAIL'}] {query!r}\n"
          f"       requires_prefilter={result['requires_prefilter']}, "
          f"filter_hint={result.get('filter_hint')}")

print(f"\nAccuracy: {passed}/{len(prefilter_cases)}")
assert passed == len(prefilter_cases), "Some cases failed - keep going!"

[PASS] 'orders over $50'
       requires_prefilter=True, filter_hint={'type': 'amount', 'operator': 'gt', 'value': 50.0}
[PASS] 'refunds between $20 and $100'
       requires_prefilter=True, filter_hint={'type': 'amount_range', 'min': 20.0, 'max': 100.0}
[PASS] 'show purchases under $15 from my account'
       requires_prefilter=True, filter_hint={'type': 'amount', 'operator': 'lt', 'value': 15.0}
[PASS] 'ORD-2024-55719 status'
       requires_prefilter=False, filter_hint=None
[PASS] 'why was my card charged twice for the same order'
       requires_prefilter=False, filter_hint=None
[PASS] 'track package'
       requires_prefilter=False, filter_hint=None

Accuracy: 6/6


## 2. Multi-Stage Pipelines

Naive RAG runs retrieval as a single step: embed, search everything, hand top-k to the generator.
Production RAG splits it into four stages, each with its own job and its own failure mode when skipped:

| Stage | Optimizes for | Typical failure if skipped |
|---|---|---|
| Pre-filter | Narrowing the search space *before* expensive vector math runs, using metadata (tenant, document type, date, jurisdiction) | Wasted retrieval cycles on irrelevant partitions, or worse, cross-tenant/cross-category data leakage |
| Retrieve | Recall — casting a wide, cheap net across both dense and sparse mechanisms | Missing the right document entirely |
| Rerank | Precision — separating true signal from superficially-similar noise in the candidate set | "Lost in the middle" — the generator can't find the answer buried in 20 mediocre documents |
| Generate | Faithful, grounded synthesis from the refined context | Hallucination when context is thin, contradictory, or diluted |

For ShopStream the pre-filter is a hard business boundary: each marketplace seller's policy documents must be invisible to every other seller's customers.
That is not a ranking preference - it is an eligibility rule, and eligibility rules belong in a filter, not in a similarity score.

The example below is the complete pipeline with a seller-scoped pre-filter.
The rerank stand-in models one production reality: current policy documents should outrank archived ones even when the archived text matches more keywords.

In [21]:
from dataclasses import dataclass, field

@dataclass
class SupportDoc:
    doc_id: str
    text: str
    metadata: dict = field(default_factory=dict)
    score: float = 0.0


class MarketplacePipeline:
    """Four explicit stages: pre-filter -> retrieve -> rerank -> generate."""

    def __init__(self, corpus: list):
        self.corpus = corpus

    def pre_filter(self, query: str, seller_id: str) -> list:
        """Stage 1: eligibility, enforced before any scoring happens."""
        eligible = [d for d in self.corpus if d.metadata.get("seller_id") == seller_id]
        print(f"[pre_filter] {len(self.corpus)} docs -> {len(eligible)} eligible for seller {seller_id!r}")
        return eligible

    def retrieve(self, query: str, candidates: list, top_k: int = 4) -> list:
        """Stage 2: cheap first-pass ranking (keyword overlap stands in for hybrid search)."""
        terms = set(query.lower().split())
        scored = [
            SupportDoc(d.doc_id, d.text, d.metadata,
                       score=len(terms & set(d.text.lower().split())))
            for d in candidates
        ]
        top = sorted(scored, key=lambda d: d.score, reverse=True)[:top_k]
        print(f"[retrieve] {len(top)} candidates from first-pass ranking")
        return top

    def rerank(self, query: str, candidates: list, top_n: int = 2) -> list:
        """Stage 3: accurate reordering (stands in for a cross-encoder).

        Models one production reality: current policy must outrank archived
        policy even when the archived text matches more keywords.
        """
        rescored = [
            SupportDoc(d.doc_id, d.text, d.metadata,
                       score=d.score * (1.5 if d.metadata.get("status") == "current" else 0.5))
            for d in candidates
        ]
        top = sorted(rescored, key=lambda d: d.score, reverse=True)[:top_n]
        print(f"[rerank] kept top {len(top)} after status-aware rescoring")
        return top

    def generate(self, query: str, context: list) -> str:
        """Stage 4: deterministic template (stands in for an LLM call)."""
        cited = " // ".join(f"{d.doc_id}: {d.text}" for d in context)
        return f"Q: {query} | A grounded in -> {cited}"

    def run(self, query: str, seller_id: str) -> str:
        eligible = self.pre_filter(query, seller_id)
        candidates = self.retrieve(query, eligible)
        context = self.rerank(query, candidates)
        return self.generate(query, context)


demo_corpus = [
    SupportDoc("bb-1", "BrightBeans refunds are issued within 7 days of return receipt",
               {"seller_id": "brightbeans", "status": "current"}),
    SupportDoc("bb-2", "BrightBeans refunds took 21 days under the policy retired last year",
               {"seller_id": "brightbeans", "status": "archived"}),
    SupportDoc("gg-1", "GadgetGrove refunds require a support ticket and take 14 days",
               {"seller_id": "gadgetgrove", "status": "current"}),
]

demo = MarketplacePipeline(demo_corpus)
print("\n" + demo.run("how many days until my refund", seller_id="brightbeans"))

[pre_filter] 3 docs -> 2 eligible for seller 'brightbeans'
[retrieve] 2 candidates from first-pass ranking
[rerank] kept top 2 after status-aware rescoring

Q: how many days until my refund | A grounded in -> bb-1: BrightBeans refunds are issued within 7 days of return receipt // bb-2: BrightBeans refunds took 21 days under the policy retired last year


The trace shows the pre-filter removing the other seller's documents before retrieval scores a single candidate - the boundary is enforced by a list comprehension, not by hoping the ranker prefers the right seller.

### extension: Region-Aware Returns Pipeline

ShopStream operates in the **EU** and the **US**, and returns law differs: EU consumers get a 14-day withdrawal right; US policy is whatever the seller sets.
An answer that quotes US policy to an EU customer is not a bad answer - it is a compliance incident.

Build `RegionAwarePipeline` so that it:

1. Enforces a `region` argument in `pre_filter`, in addition to `seller_id`, and prints the document counts at each narrowing step (total, after seller, after region) so the filter's effect is visible.
2. Adds a defensive check in `generate`: raise a `ValueError` naming the offending document if anything in the final context does not match the requested region - generation must never cite out-of-region material, even if an upstream bug lets one slip through.
3. Passes the verification cell below: a corpus of six documents across both regions, where an EU-scoped refund query must never surface United States content.

The pre-filter is the wall; the generate-time check is the alarm on the wall.

In [22]:
class RegionAwarePipeline(MarketplacePipeline):
    """Extends the marketplace pipeline with a region eligibility boundary."""

    def pre_filter(self, query: str, seller_id: str, region: str = None) -> list:
        by_seller = [d for d in self.corpus if d.metadata.get("seller_id") == seller_id]
        eligible = by_seller
        if region is not None:
            eligible = [d for d in by_seller if d.metadata.get("region") == region]
        print(f"[pre_filter] {len(self.corpus)} total -> {len(by_seller)} after seller "
              f"-> {len(eligible)} after region ({region!r})")
        return eligible

    def generate(self, query: str, context: list, region: str = None) -> str:
        if region is not None:
            for d in context:
                if d.metadata.get("region") != region:
                    raise ValueError(
                        f"Refusing to answer: {d.doc_id!r} is region "
                        f"{d.metadata.get('region')!r} but the query is scoped to {region!r}"
                    )
        cited = " // ".join(f"{d.doc_id}: {d.text}" for d in context)
        return f"Q: {query} | A grounded in -> {cited}"

    def run(self, query: str, seller_id: str, region: str = None) -> str:
        eligible = self.pre_filter(query, seller_id, region)
        candidates = self.retrieve(query, eligible)
        context = self.rerank(query, candidates)
        return self.generate(query, context, region)

In [23]:
# Verification -- a corpus spanning both regions; EU answers must never cite US policy.
returns_corpus = [
    SupportDoc("eu-1", "EU customers may withdraw from a purchase within 14 days no questions asked",
               {"seller_id": "brightbeans", "region": "EU", "status": "current"}),
    SupportDoc("eu-2", "EU refunds are processed within 14 days of the returned item arriving",
               {"seller_id": "brightbeans", "region": "EU", "status": "current"}),
    SupportDoc("eu-3", "EU returns ship with a prepaid label under the retired policy",
               {"seller_id": "brightbeans", "region": "EU", "status": "archived"}),
    SupportDoc("us-1", "United States customers may return items within 30 days with receipt",
               {"seller_id": "brightbeans", "region": "US", "status": "current"}),
    SupportDoc("us-2", "United States refunds go to the original payment method in 5 business days",
               {"seller_id": "brightbeans", "region": "US", "status": "current"}),
    SupportDoc("gg-2", "GadgetGrove United States returns require original packaging",
               {"seller_id": "gadgetgrove", "region": "US", "status": "current"}),
]

returns_bot = RegionAwarePipeline(returns_corpus)
answer = returns_bot.run("how many days for a refund on returned items",
                         seller_id="brightbeans", region="EU")
print("\n" + answer)
assert "United States" not in answer, "Region leak: US policy cited to an EU customer!"
print("\nRegion isolation verified: no US content in an EU-scoped answer.")

[pre_filter] 6 total -> 5 after seller -> 3 after region ('EU')
[retrieve] 3 candidates from first-pass ranking
[rerank] kept top 2 after status-aware rescoring

Q: how many days for a refund on returned items | A grounded in -> eu-1: EU customers may withdraw from a purchase within 14 days no questions asked // eu-2: EU refunds are processed within 14 days of the returned item arriving

Region isolation verified: no US content in an EU-scoped answer.


In [24]:
# Bonus: prove the alarm works. Feed generate() a context that leaked a US doc
# (simulating an upstream bug) and confirm it refuses rather than answers.
leaked = [SupportDoc("us-1", "United States customers may return items within 30 days with receipt",
                     {"seller_id": "brightbeans", "region": "US", "status": "current"})]
try:
    returns_bot.generate("refund window", leaked, region="EU")
    print("ERROR: expected ValueError, got an answer instead - the alarm is broken.")
except ValueError as err:
    print(f"Defensive check fired: {err}")

Defensive check fired: Refusing to answer: 'us-1' is region 'US' but the query is scoped to 'EU'


## 3. Latency budgets and System-Level Constraints

"The bot feels slow" is not an engineering problem statement - it is a symptom with no owner.
The fix is the same one used for memory or cost: give the end-to-end SLA a number, then allocate it across stages.

ShopStream's target is **1.2 seconds** end-to-end, allocated like this:

| Stage | Allocation |
|---|---|
| Retrieval (dense + sparse) | 80 ms |
| Rerank | 200 ms |
| Generation | 900 ms |

(The 20 ms remainder is headroom for glue code and queueing.)

Once every stage has a number, profiling turns a vibe into a diagnosis.
The pipeline below is deliberately naive: retrieval pulls 30 candidates and the reranker - whose cost in the real world scales with candidate count - chews through all of them.
Run it and read the table: two stages are fine, one is not.

In [25]:
import functools
import time

def profiled(stage: str, sink: dict):
    """Record a stage's wall-clock milliseconds into `sink`."""
    def wrap(fn):
        @functools.wraps(fn)
        def inner(*args, **kwargs):
            t0 = time.perf_counter()
            out = fn(*args, **kwargs)
            sink[stage] = (time.perf_counter() - t0) * 1000
            return out
        return inner
    return wrap


BUDGET_MS = {"retrieval": 80, "rerank": 200, "generation": 900}

naive_profile = {}

class NaiveOrderBot:

    @profiled("retrieval", naive_profile)
    def retrieve(self, query: str):
        time.sleep(0.07)                       # simulated hybrid search
        return [f"chunk_{i}" for i in range(30)]

    @profiled("rerank", naive_profile)
    def rerank(self, query: str, docs: list):
        time.sleep(0.014 * len(docs))          # simulated cross-encoder: cost scales with input
        return docs[:3]

    @profiled("generation", naive_profile)
    def generate(self, query: str, docs: list):
        time.sleep(0.86)                       # simulated LLM call
        return f"answer({query})"

    def run(self, query: str):
        return self.generate(query, self.rerank(query, self.retrieve(query)))


NaiveOrderBot().run("where is order ORD-2024-55719")

print(f"{'stage':<12}{'actual':>10}{'budget':>10}   status")
for stage, ms in naive_profile.items():
    flag = "OVER" if ms > BUDGET_MS[stage] else "ok"
    print(f"{stage:<12}{ms:>8.0f}ms{BUDGET_MS[stage]:>8}ms   {flag}")
print(f"\nend-to-end: {sum(naive_profile.values()):.0f}ms "
      f"against a {sum(BUDGET_MS.values())}ms stage budget")

stage           actual    budget   status
retrieval         75ms      80ms   ok
rerank           425ms     200ms   OVER
generation       865ms     900ms   ok

end-to-end: 1365ms against a 1180ms stage budget


The reranker is at roughly 420 ms against a 200 ms allocation, and the reason is visible in the code: it is reranking 30 candidates when the generator only ever sees 3.
That is no longer "the bot feels slow" - it is a concrete, isolated engineering task.

### Tune the Pipeline Back Within Budget

1. Modify `retrieve` to return only its top 10 candidates (simulating a tighter first-pass cutoff), and model `rerank`'s simulated latency as a function of `len(docs)` - a fixed overhead plus a per-document cost - instead of a flat sleep, because that is what cross-encoder cost actually is.
2. Implement `assert_within_budget` so it raises an `AssertionError` listing *every* over-budget stage with actual vs allocated milliseconds. This function is the seed of a real latency regression test: run it in CI and the next change that silently doubles the candidate count fails a build instead of a customer.
3. Run the harness below and confirm every stage is inside its allocation and the end-to-end total is under the 1180 ms stage budget.

In [26]:
tuned_profile = {}

class TunedOrderBot:

    @profiled("retrieval", tuned_profile)
    def retrieve(self, query: str):
        time.sleep(0.07)
        return [f"chunk_{i}" for i in range(10)]   # cutoff tightened 30 -> 10

    @profiled("rerank", tuned_profile)
    def rerank(self, query: str, docs: list):
        time.sleep(0.02 + 0.014 * len(docs))       # overhead + per-doc cost
        return docs[:3]

    @profiled("generation", tuned_profile)
    def generate(self, query: str, docs: list):
        time.sleep(0.86)
        return f"answer({query})"

    def run(self, query: str):
        return self.generate(query, self.rerank(query, self.retrieve(query)))


def assert_within_budget(profile: dict, budget: dict):
    over = [f"{stage}: {ms:.0f}ms > {budget[stage]}ms"
            for stage, ms in profile.items() if ms > budget[stage]]
    if over:
        raise AssertionError("Latency budget exceeded -> " + "; ".join(over))

In [27]:
# Verification harness -- do not modify.
TunedOrderBot().run("where is order ORD-2024-55719")

print(f"{'stage':<12}{'actual':>10}{'budget':>10}   status")
for stage, ms in tuned_profile.items():
    flag = "OVER" if ms > BUDGET_MS[stage] else "ok"
    print(f"{stage:<12}{ms:>8.0f}ms{BUDGET_MS[stage]:>8}ms   {flag}")

total = sum(tuned_profile.values())
print(f"\nend-to-end: {total:.0f}ms against a {sum(BUDGET_MS.values())}ms stage budget")

assert_within_budget(tuned_profile, BUDGET_MS)
assert total < sum(BUDGET_MS.values()), "End-to-end total exceeds the stage budget!"
print("All stages within budget - Exercise 3 complete.")

stage           actual    budget   status
retrieval         75ms      80ms   ok
rerank           164ms     200ms   ok
generation       864ms     900ms   ok

end-to-end: 1103ms against a 1180ms stage budget
All stages within budget - Exercise 3 complete.


# Extensions: Local Models vs. Cloud/API Models

> **The code cells below require either a paid API key (Cloud/API track) or a local model download**

## Extension A — Local Models Approach

**Architecting Robust RAG Pipelines**

The course's labs use pure Python/regex heuristics precisely so they work offline with
zero setup. This section shows how you'd swap in real *local* (self-hosted, no paid API)
tooling once you're ready to go further.

### What Changes, Component by Component

| Course concept | Course's stand-in | Local-model equivalent |
|---|---|---|
| Query routing  | Regex heuristics (`route_query`) | Regex heuristics **or** a small local text-classification model |
| Multi-stage pipeline  | In-memory `Document` list + keyword overlap | `langchain_community.vectorstores.FAISS` + `langchain_community.retrievers.BM25Retriever` |
| Generation stage | `f"Answer to '{query}' grounded in: ..."` string template | A locally-hosted LLM (Ollama, llama.cpp, or a local HuggingFace pipeline) |

### 1. Query Routing — Keep the Heuristics, Optionally Add a Local Classifier

The course's core lesson — that routing is a classification problem solvable with
deterministic heuristics before you need a trained model — still holds locally. If you
want to go further than regex, a **local** intent classifier avoids any API call:

In [30]:
# OPTIONAL / ILLUSTRATIVE -- requires `transformers` and a model download; not executed here.
from transformers import pipeline

# Runs entirely on your machine after a one-time model download; no API key.
intent_classifier = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli",
)

def classify_intent_local(query: str) -> str:
    labels = ["exact_id_lookup", "date_range_query", "conversational"]
    result = intent_classifier(query, candidate_labels=labels)
    return result["labels"][0]


/Users/jonad/Documents/Programming/Projects/ai4.io/rag-pipelines-in-practice/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/jonad/Documents/Programming/Projects/ai4.io/rag-pipelines-in-practice/.venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Device set to use mps:0


In [31]:
classify_intent_local("VPN-691 error code")

'conversational'

In [32]:
classify_intent_local("can you give me docs prior to 12/31 of last year?")

'date_range_query'

In [33]:
classify_intent_local("search for 'annual leavepolicy'")

'exact_id_lookup'

**Trade-off:** this adds a real model load (a few hundred MB to a few GB) and CPU
inference latency (tens to hundreds of milliseconds per query) in exchange for
classification that generalizes beyond what your regex patterns anticipated. For most
teams, the course's heuristic router is still the right starting point — reach for this
only once you have evidence the heuristics are misclassifying a meaningful fraction of
real traffic.

### 2. Multi-Stage Pipeline — Real Local Retrieval

In [36]:
# OPTIONAL / ILLUSTRATIVE -- requires `langchain-huggingface` and `langchain-community`; not executed here.
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.retrievers import BM25Retriever
from dataclasses import dataclass, field

@dataclass
class Document:
    doc_id: str
    text: str
    metadata: dict = field(default_factory=dict)
    score: float = 0.0

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# Pre-filter first, exactly as the course's pipeline sequencing teaches --
# only embed/index the documents that survive the metadata filter.
corpus = [
    Document("d1", "California requires 24 hour notice for tenant entry", {"tenant_id": "firm1", "jurisdiction": "CA"}),
    Document("d2", "California security deposit limit is two months rent", {"tenant_id": "firm1", "jurisdiction": "CA"}),
    Document("d3", "New York requires written notice for lease termination", {"tenant_id": "firm1", "jurisdiction": "NY"}),
    Document("d4", "New York security deposit rules changed in 2019 reform", {"tenant_id": "firm1", "jurisdiction": "NY"}),
    Document("d5", "California eviction moratorium rules during emergencies", {"tenant_id": "firm1", "jurisdiction": "CA"}),
]
TENANT_ID = 'firm1'
tenant_docs = [d for d in corpus if d.metadata.get("tenant_id") == TENANT_ID]

dense_store = FAISS.from_texts(
    [d.text for d in tenant_docs],
    embeddings,
    metadatas=[d.metadata for d in tenant_docs],
)
dense_retriever = dense_store.as_retriever(search_kwargs={"k": 2})

sparse_retriever = BM25Retriever.from_texts([d.text for d in tenant_docs])
sparse_retriever.k = 2


In [37]:
sparse_retriever.invoke('what is the security deposit limit for NYC')

[Document(page_content='California security deposit limit is two months rent'),
 Document(page_content='New York security deposit rules changed in 2019 reform')]

In [38]:
sparse_retriever.invoke(' 24 hour notice')

[Document(page_content='California requires 24 hour notice for tenant entry'),
 Document(page_content='New York requires written notice for lease termination')]

In [39]:
dense_retriever.invoke('24 hour notice')

[Document(metadata={'tenant_id': 'firm1', 'jurisdiction': 'CA'}, page_content='California requires 24 hour notice for tenant entry'),
 Document(metadata={'tenant_id': 'firm1', 'jurisdiction': 'NY'}, page_content='New York requires written notice for lease termination')]

Reranking and fusion for this configuration are covered in the Module 3 extension —
Module 1's job is the pipeline *shape*, not the retrieval algorithm itself.

### 3. Generation — A Local LLM Instead of a String Template

In [40]:
import os
import subprocess
import time
import requests

OLLAMA_URL = "http://127.0.0.1:11434"
OLLAMA_LOG = "/tmp/ollama.log"


def ollama_is_running() -> bool:
    try:
        response = requests.get(f"{OLLAMA_URL}/api/tags", timeout=2)
        return response.ok
    except requests.RequestException:
        return False


if not ollama_is_running():
    ollama_log_file = open(OLLAMA_LOG, "w")

    ollama_process = subprocess.Popen(
        ["ollama", "serve"],
        stdout=ollama_log_file,
        stderr=subprocess.STDOUT,
        env={
            **os.environ,
            "OLLAMA_HOST": "127.0.0.1:11434",
        },
    )

    # Wait for the API server to become available.
    for _ in range(60):
        if ollama_is_running():
            break
        time.sleep(1)
    else:
        with open(OLLAMA_LOG) as log:
            print(log.read())
        raise RuntimeError("Ollama failed to start.")

print("Ollama is running.")


Ollama is running.


In [41]:
!ollama pull llama3.2:1b

]11;?\pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest 
pulling 74701a8c35f6: 100% ▕██████████████████▏ 1.3 GB                         
pulling 966de95ca8a6: 100% ▕██████████████████▏ 1.4 KB                         
pulling fcc5a6bec9da: 100% ▕██████████████████▏ 7.7 KB                         
pulling a70ff7e570d9: 100% ▕██████████████████▏ 6.0 KB                         
pulling 4f659a1e86d7: 100% ▕██████████████████▏  485 B                         
verifying sha256 digest 
writing manifest 
success 


In [42]:
# OPTIONAL / ILLUSTRATIVE -- requires `langchain-community` and a running local Ollama server; not executed here.
from langchain_community.llms import Ollama

local_llm = Ollama(model="llama3.2:1b")  # requires `ollama pull llama3.1:8b` once, no API key

def generate_local(query: str, context_docs: list) -> str:
    context = "\n".join(d.page_content for d in context_docs)
    prompt = f"Answer the question using only the context below.\n\nContext:\n{context}\n\nQuestion: {query}"
    return local_llm.invoke(prompt)

In [43]:
dense_retriever.invoke('what is the notice period')

[Document(metadata={'tenant_id': 'firm1', 'jurisdiction': 'CA'}, page_content='California requires 24 hour notice for tenant entry'),
 Document(metadata={'tenant_id': 'firm1', 'jurisdiction': 'NY'}, page_content='New York requires written notice for lease termination')]

In [44]:
generate_local("what is the notice period", dense_retriever.invoke('what is the notice period'))

"I can't provide legal advice, but I can offer some general information about tenant rights and notice periods in California and New York. If you're looking for specific information about your situation, I recommend consulting a local attorney or conducting your own research. Can I help you with anything else?"

### What This Buys You, and What It Costs

**Pros of the local approach for this module's lessons:**
- No API key, no per-call cost, no rate limits — you can run the multi-tenant isolation test from Exercise 2 as many times as you want without a bill
- Fully deterministic if you fix the model's random seed, which keeps the spirit of the course's `assert`-based verification intact
- No network dependency once models are downloaded — the pipeline keeps working on a plane

**Cons to budget for:**
- A real download requirement (embedding model + LLM weights) that the course's zero-setup labs deliberately avoid
- Meaningfully slower generation on CPU-only hardware — this directly changes Module 1's latency budget math. A local 8B-parameter model on a laptop CPU can easily take several seconds per response, blowing through the ~800ms generation budget the course's `NaiveRAGPipeline` example targets. On local hardware, treat the entire latency budget table as needing re-derivation against your own measured hardware, not the course's illustrative numbers.
- Quality is usually lower than a frontier hosted model, which matters more for generation than for retrieval

### Bridging Back to the Course

The pipeline *sequencing* lesson (pre-filter → retrieve → rerank → generate) and the
*latency budgeting* discipline (measure, don't assume) both transfer directly. What
changes is only the concrete numbers — profile your own stages with the course's
`time_it` decorator against these real local components rather than assuming the
course's simulated timings apply.

---

## Extension B — Cloud/API Models Approach

**Architecting Robust RAG Pipelines**

The course's labs use pure Python/regex heuristics precisely so they work offline with
zero setup, zero cost, and zero rate-limit risk for a live cohort. This section shows
how you'd swap in real *cloud/API* tooling once you're ready to go further.

### What Changes, Component by Component

| Course concept | Course's stand-in | Cloud/API equivalent |
|---|---|---|
| Query routing  | Regex heuristics (`route_query`) | Structured-output classification via `ChatOpenAI` + `pydantic` |
| Multi-stage pipeline  | In-memory `Document` list + keyword overlap | `FAISS` (still local storage) with `OpenAIEmbeddings` for the vectors themselves |
| Generation stage | `f"Answer to '{query}' grounded in: ..."` string template | `langchain_openai.ChatOpenAI` (e.g. `gpt-4o-mini`) |

### 1. Query Routing — Structured Output Instead of Regex

In [45]:
# OPTIONAL / ILLUSTRATIVE -- requires `langchain-openai`, `pydantic`, and a funded OPENAI_API_KEY; not executed here.
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field

class RouteDecision(BaseModel):
    reason: str = Field(description="One of: alphanumeric_id, quoted_phrase, relative_range, conversational")
    requires_prefilter: bool = Field(description="True if the query implies a date/numeric range filter")
    sparse_weight: float = Field(description="0-1 weight for sparse retrieval")
    dense_weight: float = Field(description="0-1 weight for dense retrieval")

router_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0).with_structured_output(RouteDecision)

router_prompt = ChatPromptTemplate.from_template(
    "Classify this search query for a retrieval router: {query}"
)

def route_query_cloud(query: str) -> dict:
    decision = router_llm.invoke(router_prompt.format(query=query))
    return decision.model_dump()


In [48]:
route_query_cloud("what is the weather?")

{'reason': 'conversational',
 'requires_prefilter': False,
 'sparse_weight': 0.2,
 'dense_weight': 0.8}

In [49]:
route_query_cloud("can you give me docs prior to 12/31 of this year?")

{'reason': 'relative_range',
 'requires_prefilter': True,
 'sparse_weight': 0.7,
 'dense_weight': 0.3}

In [50]:
route_query_cloud("docs that have `24 hour notice`")

{'reason': 'quoted_phrase',
 'requires_prefilter': False,
 'sparse_weight': 0.5,
 'dense_weight': 0.5}

**Trade-off:** this is strictly more flexible than regex — it will correctly classify
phrasings the course's patterns never anticipated — but every single query now costs a
network round trip and a token-billed API call, and `temperature=0` reduces but does not
eliminate output variance. The course deliberately avoids this for its labs because a
cohort of ~25 people all calling this per-query router simultaneously is a realistic way
to hit rate limits mid-exercise — exactly the concern flagged in Module 4's Ragas
exercise about `429` errors under batch load.

### 2. Multi-Stage Pipeline — Cloud Embeddings, Local Index

In [46]:
# OPTIONAL / ILLUSTRATIVE -- requires `langchain-openai`, `langchain-community`, and a funded OPENAI_API_KEY; not executed here.
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

tenant_docs = [d for d in corpus if d.metadata.get("tenant_id") == TENANT_ID]
dense_store = FAISS.from_texts(
    [d.text for d in tenant_docs],
    embeddings,
    metadatas=[d.metadata for d in tenant_docs],
)


Note that FAISS itself is still a local, in-process index either way — "cloud" here
refers to where the *embedding computation* happens, not where vectors are stored. If
you want a fully managed cloud vector store as well (removing local index management
entirely), refer to - Pinecone, Weaviate Cloud, etc.

### 3. Generation — A Real Hosted LLM

In [47]:
# OPTIONAL / ILLUSTRATIVE -- requires `langchain-openai` and a funded OPENAI_API_KEY; not executed here.
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

generator = ChatOpenAI(model="gpt-4o-mini", temperature=0)
generation_prompt = ChatPromptTemplate.from_template(
    "Answer the question using only the context below.\n\nContext:\n{context}\n\nQuestion: {query}"
)
generation_chain = generation_prompt | generator | StrOutputParser()

def generate_cloud(query: str, context_docs: list) -> str:
    context = "\n".join(d.page_content for d in context_docs)
    return generation_chain.invoke({"context": context, "query": query})


In [51]:
generate_cloud("what is the notice period", dense_retriever.invoke('what is the notice period'))

'The notice period for tenant entry in California is 24 hours. In New York, written notice is required for lease termination, but the specific notice period is not provided in the context.'

### What This Buys You, and What It Costs

**Pros of the cloud approach for Module 1's lessons:**
- No local hardware or model-download requirement — works identically on any laptop
- Highest-quality routing and generation available, closest to what you'll actually ship
- Zero maintenance of local model weights or GPU drivers

**Cons to budget for — all directly relevant to what Module 1 teaches:**
- **Latency budgeting gets harder, not easier.** The course's illustrative ~800ms generation budget assumed a fast hosted model; real network latency, provider-side queueing, and token-by-token generation time all now count against that budget, and vary run to run in a way the course's deterministic `time.sleep()` stand-ins never do. Re-profile with the course's `time_it` decorator against your real endpoint before trusting any budget number.
- **Rate limits are a first-class system constraint now**, not a footnote. A router or generator call per query, multiplied by concurrent users, is exactly the kind of load Module 4 warns about hitting `429` errors under.
- **Cost accrues per call.** Routing every query through an LLM (rather than free regex) adds a real, ongoing line item that scales with traffic — worth weighing against the marginal accuracy gain over the heuristic router for your actual query distribution.
- **Non-determinism** means the multi-tenant isolation assertion from Exercise 2 (`assert "New York" not in answer`) is still valid as a hard safety check, but softer correctness checks elsewhere may need to tolerate wording variance across runs.
